# 07 - Neural Network Forecasting Engine

Goal: train a lightweight neural-network baseline for next-day return forecasting and compare it with classical models.

This uses `sklearn.neural_network.MLPRegressor` to keep the project reproducible on normal laptops. A heavier LSTM/GRU model can be added later if the competition environment allows TensorFlow or PyTorch.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import joblib
import pandas as pd
import plotly.express as px

from invest_intel.data import load_processed
from invest_intel.features import drop_modeling_na, modeling_feature_columns
from invest_intel.modeling import train_neural_network_return_model

In [2]:
features = load_processed('nifty50_features.csv')
feature_cols = modeling_feature_columns(features)
model_df = drop_modeling_na(features, 'future_return_1d', feature_cols)
model_df.shape, len(feature_cols)

((119695, 52), 40)

In [3]:
nn_result = train_neural_network_return_model(
    model_df,
    feature_cols,
    target_col='future_return_1d',
    test_start_date='2019-01-01',
)
nn_result.metrics

{'mae': 0.017051637929005225,
 'rmse': 0.026495995889925808,
 'r2': -0.02751946703521657,
 'directional_accuracy': 0.49815759637188206}

In [4]:
predictions = nn_result.predictions.copy()
predictions['actual_direction'] = predictions['future_return_1d'] > 0
predictions['predicted_direction'] = predictions['prediction'] > 0
predictions.head()

,date,symbol,future_return_1d,prediction,actual_direction,predicted_direction
1623,2019-01-01,ADANIPORTS,-0.019794,0.001255,False,True
1624,2019-01-02,ADANIPORTS,-0.008392,0.000654,False,True
1625,2019-01-03,ADANIPORTS,0.010711,0.001121,True,True
1626,2019-01-04,ADANIPORTS,-0.001308,0.002001,False,True
1627,2019-01-07,ADANIPORTS,-0.007599,0.001370,False,True


In [5]:
sample = predictions[predictions['symbol'] == 'RELIANCE'].tail(250)
px.line(sample, x='date', y=['future_return_1d', 'prediction'], title='RELIANCE actual vs neural-network predicted next-day return')

In [6]:
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump({'model': nn_result.estimator, 'feature_cols': feature_cols, 'target': 'future_return_1d'}, ROOT / 'models' / 'stock_return_1d_neural_network.joblib')

['e:\\Projects\\cult_invest_intel_ml\\models\\stock_return_1d_neural_network.joblib']